# B4: attention scores, causal mask and softmax

**Goal:** understand and verify `softmax(Q @ K.T / sqrt(D) + mask) @ V`.

Think of each query as a question, keys as possible matches, and values as the content to mix. These are analogies, not fixed roles learned by individual heads.

**Spec:** Q `(Nq, H, D)`, K/V `(Nk, H, D)`, allowed mask `(Nq, Nk)` -> output `(Nq, H, D)` and weights `(H, Nq, Nk)`. True means visible. Every row must have at least one visible key. FP16/BF16 use FP32 computation and return the input output dtype.

B4 uses equal head counts. HF supplies the head expansion only for the real-model check; we build GQA ourselves in B5.

Run top to bottom in **Colab with a T4**. Tests run first. Do not mark this lesson complete until the tests and reference comparisons pass.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib, math
import torch
import tinyserve.models.attention, tinyserve.probe, tinyserve.plots
for module in (tinyserve.models.attention, tinyserve.probe, tinyserve.plots):
    importlib.reload(module)
from tinyserve.models.attention import causal_mask, reference_attention
from tinyserve.envinfo import collect_env, save_result
from tinyserve.plots import plot_attention
from IPython.display import Image, display

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

The tests define the contract: hand math, an independent float64 formula, PyTorch attention, zero future weights, row sums, packed isolation, RoPE integration and precision checks. A small randomly initialized HF model tests the real-activation probe without downloading weights.

A failed test stops this notebook. Read any skips: SDPA, HF, plotting and CUDA checks should be available in this Colab environment.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b4_attention.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. By hand: two tokens, one head, one dimension

Q = `[1, 1]`, K = `[0, ln(3)]`, V = `[10, 20]`.

- Token 0 cannot see token 1: scores `[0, -inf]` -> weights `[1, 0]` -> output `10`.
- Token 1 sees both: scores `[0, ln(3)]` -> exponentials `[1, 3]` -> weights `[0.25, 0.75]` -> output `17.5`.

The weighted sum is **25% of 10 + 75% of 20**. This output is a vector to pass to the next operation, not a predicted token yet.

In [ ]:
hand_query = torch.ones(2, 1, 1, device=DEVICE)
hand_key = torch.tensor([[[0.0]], [[math.log(3.0)]]], device=DEVICE)
hand_value = torch.tensor([[[10.0]], [[20.0]]], device=DEVICE)
hand_mask = causal_mask(torch.arange(2, device=DEVICE))
hand_output, hand_weights = reference_attention(hand_query, hand_key, hand_value, hand_mask)
print("allowed mask:", hand_mask.tolist())
print("weights:", hand_weights[0].tolist())
print("output:", hand_output[:, 0, 0].tolist())
torch.testing.assert_close(hand_output[:, 0, 0], torch.tensor([10.0, 17.5], device=DEVICE), rtol=0, atol=1e-6)


## 3. Realistic shapes: four tokens, two heads, D=64

Q/K/V each have shape `(4, 2, 64)`. Each head gets a `(4, 4)` score matrix. Divide by `sqrt(64) = 8`, mask the future, then softmax over keys.

Why divide? For roughly independent unit-variance components, a D-term dot product has variance about D. Scaling by sqrt(D) prevents larger heads from automatically producing much sharper softmax distributions.

Rows ask; columns are looked at. The heatmap must have zeros above its diagonal. These are synthetic vectors, not the model's learned attention.

In [ ]:
demo_generator = torch.Generator(device=DEVICE).manual_seed(42)
demo_query, demo_key, demo_value = [torch.randn(4, 2, 64, device=DEVICE, generator=demo_generator) for _ in range(3)]
demo_mask = causal_mask(torch.arange(4, device=DEVICE))
demo_output, demo_weights = reference_attention(demo_query, demo_key, demo_value, demo_mask)
print("output shape:", tuple(demo_output.shape), "weights shape:", tuple(demo_weights.shape))
print("head 0 weights:\n", demo_weights[0].cpu())
print("row sums:", demo_weights.sum(dim=-1).tolist())
future_weight_max = demo_weights.masked_select(~demo_mask[None]).max().item()
print("largest future weight:", future_weight_max)
assert future_weight_max == 0.0
os.makedirs("results", exist_ok=True)
plot_path = plot_attention(demo_weights[0], ["The", "capital", "of", "France"], "results/B4_attention.png")
display(Image(plot_path))


## 4. Causality and packed requests

**Causality:** change the last token's K/V drastically. Outputs for the earlier three tokens must not change.

**Isolation:** pack two two-token requests into four rows. Positions are `[0, 1, 0, 1]`, but positions alone are not enough: add sequence IDs `[0, 0, 1, 1]`. Packed outputs must match running the two requests separately.

In [ ]:
changed_key, changed_value = demo_key.clone(), demo_value.clone()
changed_key[-1] += 100
changed_value[-1] += 100
changed_output, _ = reference_attention(demo_query, changed_key, changed_value, demo_mask)
prefix_max_diff = (demo_output[:3] - changed_output[:3]).abs().max().item()
print("earlier outputs changed by:", prefix_max_diff)
assert prefix_max_diff == 0.0

packed_positions = torch.tensor([0, 1, 0, 1], device=DEVICE)
packed_sequence_ids = torch.tensor([0, 0, 1, 1], device=DEVICE)
packed_mask = causal_mask(packed_positions, packed_sequence_ids)
packed_output, _ = reference_attention(demo_query, demo_key, demo_value, packed_mask)
separate_outputs = []
for start, end in ((0, 2), (2, 4)):
    separate_output, _ = reference_attention(demo_query[start:end], demo_key[start:end], demo_value[start:end],
                                             causal_mask(packed_positions[start:end]))
    separate_outputs.append(separate_output)
separate_output = torch.cat(separate_outputs)
packed_max_diff = (packed_output - separate_output).abs().max().item()
print("packed mask:\n", packed_mask.cpu())
print("packed vs separate largest difference:", packed_max_diff)
torch.testing.assert_close(packed_output, separate_output, rtol=1e-5, atol=1e-6)


## 5. FP16 compared with FP32

Compare both computations on the **same FP16-rounded inputs**, so this measures computation/output rounding rather than input quantization. Our reference computes in FP32 and casts once at the end. FP16's final relative rounding is at most about `2^-11`, so use `rtol=atol=5e-4`.

This is a correctness reference, not the fastest attention kernel. Returning the full `(H, N, N)` weights is useful here but expensive for long prompts.

In [ ]:
half_query, half_key, half_value = [tensor.half() for tensor in (demo_query, demo_key, demo_value)]
half_output, _ = reference_attention(half_query, half_key, half_value, demo_mask)
float_output, _ = reference_attention(half_query.float(), half_key.float(), half_value.float(), demo_mask)
fp16_max_diff = (half_output.float() - float_output).abs().max().item()
print("FP16 vs FP32 largest difference:", fp16_max_diff)
torch.testing.assert_close(half_output.float(), float_output, rtol=5e-4, atol=5e-4)


## 6. Against real TinyLlama attention (FP32 parity)

Load FP32 weights (about 4.4 GB) and use HF's eager attention, not a fused kernel. Close other GPU-heavy notebook kernels if memory is low.

Capture Q/K/V projections and the **input to `o_proj`**. Apply B3's RoPE to Q/K, expand K/V using HF's helper, then run our B4 core. Compare layers 0, 10 and 21 before the output projection.

FP32 parity tolerance is `rtol=atol=1e-5`. The probe asserts it; a failure stops the notebook. This checks the attention core, not the complete decoder or generated-token parity (B7/B8).

In [ ]:
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_attention_layers

tokenizer, model = load_model(device=DEVICE, dtype=torch.float32)
model.set_attn_implementation("eager")
prompt_ids = encode(tokenizer, "The capital of France is", DEVICE)
attention_rows = compare_attention_layers(model, prompt_ids)
for row in attention_rows:
    print(f"layer {row['layer']:>2}: max |diff| {row['max_abs_diff']:.2e}, "
          f"row-sum error {row['row_sum_max_error']:.2e}, future weight {row['future_weight_max']:.1f}, shape {row['shape']}")


## 7. Save the evidence

Save measured differences, the test output and environment versions. The min/max below are the range **across tested layers**, not repeated performance trials. No speed benchmark is part of B4.

Copy the printed JSON back to the repo after the successful Colab run. The heatmap is `results/B4_attention.png`.

In [ ]:
layer_differences = [row["max_abs_diff"] for row in attention_rows]
saved_path = save_result("B4_attention", {
    "head_dim": 64,
    "hand_output": hand_output[:, 0, 0].tolist(),
    "future_weight_max": future_weight_max,
    "prefix_max_abs_diff": prefix_max_diff,
    "packed_max_abs_diff": packed_max_diff,
    "fp16_vs_fp32_max_abs_diff": fp16_max_diff,
    "real_model_dtype": "float32",
    "real_model_backend": "eager",
    "real_model_comparison": attention_rows,
    "real_model_diff_range_across_layers": {"min": min(layer_differences), "max": max(layer_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


## 8. Three interview questions

1. **Why divide by sqrt(D)?** Dot-product variance grows with D. Scaling controls the size of the scores entering softmax.
2. **Why mask before softmax?** `-inf` gets probability zero, while the visible weights still sum to 1. A token can see itself because it predicts the next token.
3. **How did you verify it?** Hand math, float64 and PyTorch references, causality, packed isolation, precision checks, then real HF activations in FP32.

**Next: B5 GQA.** Keep 32 query heads but only 4 KV heads, with each KV head serving 8 query heads. Replace HF's expansion helper with our own tested implementation.

After validation: retain the JSON and notebook outputs, update the guide/tracker, then commit and push the lesson.

In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
